# Test de la fonction d'association de station à des échangeurs

In [41]:
from datetime import date
from pathlib import Path

import folium
import json
import geopandas as gpd
import pandas as pd

CHUNK_SIZE: int = 20 #0
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75
RAYON_ECHANGEUR = 3000
BUFFER_ECHANGEUR = 100

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,7, 25)
#day = date(2026,8, 1)
#day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [19]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../saturation/data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../saturation/source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics


In [53]:
def stations_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
    distance_max: float = 3000,
) -> pd.DataFrame:

    #stations = stations[[ID_STATION, "geometry"]].to_crs(epsg=2154)
    #echangeurs = echangeurs[[ID_POOL, "geometry"]].to_crs(epsg=2154)

    result = gpd.sjoin_nearest(
        stations,
        echangeurs,
        how="inner",
        max_distance=distance_max,
        distance_col="distance_m",
    )

    result = result.loc[
        result["distance_m"] < distance_max,
        [ID_STATION, ID_POOL, "distance_m"],
    ]

    return pd.DataFrame(result).reset_index(drop=True)

In [57]:
def surfaces_echangeurs(
    stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
    liens: pd.DataFrame,
) -> gpd.GeoDataFrame:

    # Géométries
    points_stations = stations.merge(
        liens,
        on=ID_STATION,
        how="inner",
    )[[ID_POOL, "geometry"]]
    points = pd.concat([points_stations, echangeurs], ignore_index=True)

    # enveloppes convexes
    surfaces = points.dissolve(by=ID_POOL)
    surfaces["geometry"] = surfaces.geometry.convex_hull.buffer(BUFFER_ECHANGEUR)
  
    return surfaces.reset_index()

In [8]:
def init_pools(file_path: Path, only_aires: bool = True) -> gpd.GeoDataFrame:
    """Initialize pools from a geojson file."""
    #pools = pd.read_csv(pools_file).rename(columns={"id_aire": "id_pool"})
    pools = gpd.read_file(file_path).rename(columns={"id_aire": "id_pool"})
    if only_aires:
        pools = pools[pools[ID_POOL].str[:1] == "A"]
    return pools

In [34]:
dmr_file = "../saturation/data_DMR/aires_dmr_2026-08-03.geojson"
echangeurs = init_pools(dmr_file, False)[pools[ID_POOL].str[:1] == "E"]
ech_geo = gpd.GeoDataFrame(echangeurs, geometry=gpd.points_from_xy(echangeurs["longitude"], echangeurs["latitude"]), crs="EPSG:4326")
ech_geo = ech_geo[[ID_POOL, "geometry"]].to_crs(epsg=2154)
#ech_geo

In [35]:
min_power = 50
statics = read_statics(day, min_power)
statics = statics[statics["puissance_nominale"] >= min_power]
stations = statics[[ID_STATION, "latitude", "longitude"]].drop_duplicates()
stations_geo = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]), crs="EPSG:4326")
stations_geo = stations_geo[[ID_STATION, "geometry"]].to_crs(epsg=2154)
#stations_geo


In [ ]:
ech_station = gpd.sjoin_nearest(
    stations_geo,
    ech_geo,
    how="inner",
    max_distance=RAYON_ECHANGEUR,
    distance_col="distance",
)
ech_station = pd.DataFrame(ech_station[[ID_STATION, ID_POOL, "distance"]]).reset_index(drop=True)

ech_station

,id_station_itinerance,geometry,index_right,id_pool,distance
2,FREVCP000379,POINT (684026.883 6861501.701),1904,E000034,2365.691914
6,FREVZP28EE7D709C7C4189B1B9336CC95C7,POINT (570527.667 6327781.218),2246,E000376,620.571566
16,FRTSLP413867,POINT (570441.299 6324277.481),2247,E000377,148.515913
21,FRIZFPFAST738,POINT (567808.606 6907030.792),2178,E000308,377.429495
31,FRIOYP13530974,POINT (437269.546 6716912.692),2151,E000281,302.319842
...,...,...,...,...,...
65404,FRECHPCSUTC,POINT (840020.717 6637162.532),1986,E000116,1864.589166
65770,FRECHPCFFUV,POINT (526738.295 6705321.944),2102,E000232,2448.131515
66004,FRSIGPSIGE236,POINT (667565.243 6861054.628),1897,E000027,1498.653938
66187,FRLDLPLFR4325EVCP,POINT (833173.322 6539527.394),2483,E000613,393.621223


In [56]:
surf = surfaces_echangeurs(stations_geo, ech_geo, ech_station[[ID_STATION, ID_POOL]])
surf

,id_pool,geometry
0,E000001,"POLYGON ((664593.463 6876697.106, 664584.653 6..."
1,E000002,"POLYGON ((666498.004 6881648.839, 666497.522 6..."
2,E000003,"POLYGON ((667435.441 6885845.237, 667433.645 6..."
3,E000004,"POLYGON ((668155.242 6892724.739, 668154.761 6..."
4,E000005,"POLYGON ((671736.752 6900787.112, 671742.542 6..."
...,...,...
1081,E001082,"POLYGON ((991826.166 6542059.209, 991825.685 6..."
1082,E001083,"POLYGON ((991505.838 6543182.972, 991505.356 6..."
1083,E001084,"POLYGON ((990468.501 6543301.927, 990468.019 6..."
1084,E001085,"POLYGON ((600980.054 6165187.591, 600979.572 6..."


In [42]:
import webbrowser
from pathlib import Path

In [43]:
refnat = {'tiles':'CartoDB positron', 'location': [46.3, 2.3], 'zoom_start': 7}
carte = surf.explore(m=folium.Map(**refnat), color='green', name='aires', show=True)
carte.save("aires.html")
webbrowser.open(Path("aires.html").resolve().as_uri())

True